# Leaves in the joint model: keep what is predicted, release the rest
**Goal.** A model that predicts in latent space, is shaped for prediction, keeps everything up to the detail level its predictor (capacity + training steps)
has learned, and removes everything else. The recipe from the earlier runs: **rescue all content first** (reconstruction), **then fade**. The problem: after the
fade everything erodes a little (fresh decoder), because the latent prediction loss only penalises what is unpredictable and nothing rewards holding.

**The candidate fix: decoded prediction** (`dp`). The predictor's output is decoded and compared with the **real next frame**, class-weighted like the
reconstruction: `||D(P(z_ctx, a)) - x_next||`. Deleting content the predictor uses no longer pays (the pixels are still there), noise earns nothing (no prediction
can contain it), and content the predictor cannot use yet earns nothing either (it is released, by design).

**The design: one parent, every arm a fork.** The parent is plain `joint` (reconstruction + latent prediction + SIGReg, 20k steps): the availability stage. Every arm
starts from its checkpoint, so all arms share the same learned content and predictor, and differ only in what happens next (12k steps; the schedule changes over the
first 4k). `dp` introduced by a fork is cross-faded in (0 -> 1) over the same 4k steps as the reconstruction falls.

| fork | reconstruction | decoded prediction | role |
|---|---|---|---|
| `continue` | 1 | off | baseline: the joint model trained on |
| `continue + dp` | 1 | 0 -> 1 | what dp does on its own (dynamics, noise) |
| `fade` | 1 -> 0.1 | off | the control that eroded before |
| `fade + dp` | 1 -> 0.1 | 0 -> 1 | **idea 3**: dp as the consolidation of the fade |
| `fade-r0 + dp` | 1 -> 0 | 0 -> 1 | **idea 3, strict**: no reconstruction at all at the end |
| `dp only` | 0 at once | 1 at once | **idea 2**: reconstruct only what can be predicted, after availability |

Against the old design (three runs from scratch, then forks of `joint + dp`): one 20k run instead of three, dp's effect in the warmup and in the fade are no longer
confounded (dp enters exactly at the fork), and `dp only` starts from available content (its from-scratch version had no rescue stage, so it tested a different goal).

Same everywhere: ground-truth tile classes, cap 4, FiLM predictor, D = 128, seed 0. Every claim is read with decoders (`gaf/readout.py`), plus the latent's
**variance shares** (how loud each part is in the latent).

**Pre-registered expectations** (written before the results; the verdict cells apply them). "Parent" = `joint` at 20k. Noise is **released** when two of three
instruments agree: own decoder on the real next frame drops by >= 0.3; twinkle variance share falls to <= half; a fresh decoder drops by >= 0.2.
**Holds** = no drop > 0.05 in agent, inventory, facing (fresh decoder) and in agent, facing (own decoder). **Dynamics kept** = predicted move-step facing no
more than 0.05 below the parent.

* **H1, dp helps the dynamics** (`continue + dp` vs `continue`): predicted move-step facing or predicted hand at least 0.05 higher.
* **H2, idea 2** (`dp only`): **a** releases the noise; **b** holds the learned content and keeps the dynamics; **c** (the price, a risk) what the parent
  predicted poorly (mob, terrain) is lost too: own-decoder drop > 0.1.
* **H3, idea 3** (`fade + dp`, `fade-r0 + dp`): holds, releases the noise, keeps the dynamics.
* **H4, control** (`fade`): erodes (fresh-decoder facing drops by >= 0.2), as before. If it does not, the fade problem did not reproduce and H3 means little.
* **H5, release follows the predictor** (the goal's "up to the detail level"): across parts, what a fork with dp releases (parent holds - fork holds, own decoder)
  is larger where the parent predicted worse: Spearman correlation between the parent's predicted score and the release < -0.5.

In [ ]:
%pip install -q -e ..
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from gaf.env_grid import GridCfg
from gaf.train import Cfg
from gaf import analysis as A
from gaf import readout as R
from gaf import fork as F

OUT = "/workspace/results/joint_dp"
SEED, D, STEPS = 0, 128, 20_000
FORK_STEPS, FADE = 12_000, 4_000          # forks: the schedule changes over the first 4000 steps, then 8000 at the end values
DS_STEPS = 5000                           # steps of the fresh decoder of the decoder-strength readout (the same for every row)
TC_STEPS = (5000, 10000, 15000, 20000)    # checkpoints of the parent's timeline
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, floor_start=10 ** 9, eval_every=2000, eval_parts=6000, eval_parts_steps=500,
              D=D, seed=SEED, env=ENV, rec_class_power=1.0, rec_class_cap=4.0, pred_arch="film", ckpt_every=5000)
PARENT = Cfg(arm="joint", tag="dp-joint", **COMMON)      # the same run id as the old `joint` run: an existing checkpoint is reused
P_J = Path(OUT) / PARENT.run_id
# fork name -> (mode, tag, fade steps, reconstruction floor, dp weight or None)
FORKS = {"continue":      ("continue", "",      FADE, 1.0, None),
         "continue + dp": ("continue", "dp",    FADE, 1.0, 1.0),
         "fade":          ("fade",     "",      FADE, 0.1, None),
         "fade + dp":     ("fade",     "dp",    FADE, 0.1, 1.0),
         "fade-r0 + dp":  ("fade",     "dpr0",  FADE, 0.0, 1.0),
         "dp only":       ("fade",     "dponly", 1,   0.0, 1.0)}
FDIR = {n: P_J / "forks" / F.fork_name(m, t) for n, (m, t, *_) in FORKS.items()}
COLOR = dict(zip(["parent", *FORKS], A.RUN_COLORS * 2))
pd.set_option("display.precision", 3)

## Stage 1: the parent (availability)
One run from scratch in a background process (safe to re-run: finished and running runs are skipped). To stop: `kill -TERM -- -$(cat /workspace/results/joint_dp/launch_*.pid)`.

In [ ]:
A.launch([PARENT], OUT, workers=1)

In [ ]:
display(A.progress(OUT, [PARENT]))

In [ ]:
A.live_probes(OUT, {"joint": PARENT}, every=30)

**Recovery cell.** Only if a run died in its final probes (it has `ckpt_<steps>.pt` and a full `metrics.csv` but no `summary.json`).

In [ ]:
if P_J.exists() and not (P_J / "summary.json").exists() and (P_J / f"ckpt_{PARENT.steps}.pt").exists():
    print("finalizing the parent")
    A.finalize(PARENT, OUT)

## Stage 1, analysis: what the parent holds and foresees, and when
The model's own decoder applied to the **real next latent** (what the latent holds), to the **predictor's output** (what it foresees) and to the **last frame**
(the floor); a fresh decoder on the frozen latent; the latent's variance shares; and the same over the checkpoints (5k-20k). Columns are decoded tile classes
(1 = perfect, 0 = nothing, negative = worse than "decode nothing") and the facing orientation accuracy (chance 0.25; "move steps" are the clips where the last
action moved or turned the agent). The parent's **predicted** row is the detail level the forks should keep (H5).

In [ ]:
DYN_P = R.dynamics(P_J).loc["live"]
_f = P_J / "fresh_decoder.csv"  # the fresh decoder trains once; a re-run reads the file
if not _f.exists():
    R.decoder_strength(P_J, steps=DS_STEPS).loc["live"].to_csv(_f)
CON_P = pd.read_csv(_f, index_col=0).iloc[:, 0]
VS_P = R.variance_shares(P_J)
display(DYN_P.round(2))
print("fresh decoder:"); display(CON_P.to_frame("parent").T.round(2))
print("variance shares:", {k: round(v, 3) for k, v in VS_P.items()})


def timecourse(d):
    rows = {}
    have = {int(p.stem.split('_')[1]) for p in Path(d).glob('ckpt_*.pt')}
    for st in [s for s in TC_STEPS if s in have]:
        dyn = R.dynamics(d, n_eval=1024, step=st).loc["live"]
        rows[st] = {"predicted facing (move)": dyn.loc["predicted", "facing acc, move steps"], "predicted hand": dyn.loc["predicted", "hand"],
                    "predicted agent": dyn.loc["predicted", "agent"], "predicted mob": dyn.loc["predicted", "mob"],
                    "latent holds: mob": dyn.loc["actual next frame", "mob"], "latent holds: terrain": dyn.loc["actual next frame", "map"],
                    "latent holds: twinkle": dyn.loc["actual next frame", "twinkle"], "twinkle variance share": R.variance_shares(d, n=2048, step=st)["twinkle"]}
    return pd.DataFrame(rows).T


TC = timecourse(P_J)
fig, axes = plt.subplots(2, 4, figsize=(18, 7), layout="constrained")
for ax, col in zip(axes.flat, TC.columns):
    ax.plot(TC.index, TC[col], marker="o")
    ax.set(title=col, xlabel="step")
plt.show()
display(TC.round(3))

## Stage 2: the forks
Six forks of the parent's 20k checkpoint, in parallel background processes (safe to re-run: done and running forks are skipped). See the table at the top.

In [ ]:
for n, (mode, tag, fade, r_floor, dp) in FORKS.items():
    print(n, F.launch_forks(P_J, modes=(mode,), steps=FORK_STEPS, fade=fade, r_floor=r_floor, tag=tag, dp=dp))

In [ ]:
display(F.status(P_J, modes=[p.name for p in FDIR.values()]))

## Stage 2, analysis: what each fork holds, releases and foresees
For the parent and every fork: the fresh-decoder content, the own decoder on the real next latent, the decoded predicted frame, and the three measures of the noise.

In [ ]:
import json


def readouts(d, force=False):
    """All readouts of one run, cached in <run>/readouts.json: the fresh decoder (the slow part) is trained once, never again by a re-run or a display."""
    cache = Path(d) / "readouts.json"
    if cache.exists() and not force:
        return json.loads(cache.read_text())
    dyn, con, vs = R.dynamics(d).loc["live"], R.decoder_strength(d, steps=DS_STEPS).loc["live"], R.variance_shares(d)
    out = {"fresh: agent": con["agent"], "fresh: hand": con["hand"], "fresh: inventory": con["inventory"], "fresh: facing": con["facing acc"],
           "fresh: mob": con["mob"], "fresh: terrain": con["map"],
           **{f"own: {k}": dyn.loc["actual next frame", c] for k, c in (("agent", "agent"), ("hand", "hand"), ("inventory", "inventory"),
                                                                          ("facing", "facing acc"), ("mob", "mob"), ("terrain", "map"))},
           **{f"pred: {k}": dyn.loc["predicted", c] for k, c in (("agent", "agent"), ("hand", "hand"), ("inventory", "inventory"),
                                                                   ("facing (move)", "facing acc, move steps"), ("mob", "mob"), ("terrain", "map"))},
           "twinkle: own decoder": dyn.loc["actual next frame", "twinkle"], "twinkle: variance share": vs["twinkle"], "twinkle: fresh": con["twinkle"]}
    out = {k: float(v) for k, v in out.items()}
    cache.write_text(json.dumps(out))
    return out


# compute (slow once, then cached); nothing is displayed here
done = [n for n in FORKS if (FDIR[n] / "done").exists()]
for n, d in (("parent", P_J), *((n, FDIR[n]) for n in done)):
    print(n, "cached" if (d / "readouts.json").exists() else "computing ...", flush=True)
    readouts(d)

In [ ]:
# display only: reads the cache, trains nothing
S2 = pd.DataFrame({"parent": readouts(P_J), **{n: readouts(FDIR[n]) for n in done}}).T
pd.set_option("display.max_columns", None, "display.width", 250)
display(S2.round(2).T)

## Verdicts (rules written before the results)

In [ ]:
def say(label, ok, detail=""):
    print(f"{label}: {'yes' if ok else 'NO'}  {detail}")


def holds(n, ref="parent"):
    drop = {k: round(S2.loc[ref, k] - S2.loc[n, k], 2) for k in ("fresh: agent", "fresh: inventory", "fresh: facing", "own: agent", "own: facing")}
    return all(v <= 0.05 for v in drop.values()), drop


def releases(n, ref="parent"):
    b, f = S2.loc[ref], S2.loc[n]
    votes = [b["twinkle: own decoder"] - f["twinkle: own decoder"] >= 0.3, f["twinkle: variance share"] <= 0.5 * b["twinkle: variance share"],
             b["twinkle: fresh"] - f["twinkle: fresh"] >= 0.2]
    return sum(votes) >= 2, votes


def dyn_kept(n):
    return S2.loc[n, "pred: facing (move)"] >= S2.loc["parent", "pred: facing (move)"] - 0.05


g = lambda n, k: S2.loc[n, k] if n in S2.index else float("nan")
if "continue + dp" in S2.index and "continue" in S2.index:
    say("H1 dp helps the dynamics", g("continue + dp", "pred: facing (move)") >= g("continue", "pred: facing (move)") + 0.05
        or g("continue + dp", "pred: hand") >= g("continue", "pred: hand") + 0.05,
        f"facing (move) {g('continue + dp', 'pred: facing (move)'):.2f} vs {g('continue', 'pred: facing (move)'):.2f}; "
        f"hand {g('continue + dp', 'pred: hand'):.2f} vs {g('continue', 'pred: hand'):.2f}")
for n in ("dp only", "fade + dp", "fade-r0 + dp", "fade", "continue + dp"):
    if n not in S2.index:
        continue
    h, drop = holds(n)
    r, votes = releases(n)
    print(f"--- {n}")
    print("  drops of learned content vs parent:", drop)
    say("  holds", h)
    say("  releases noise (>= 2 of 3)", r, f"votes {votes}")
    say("  dynamics kept", dyn_kept(n), f"(pred facing (move) {S2.loc[n, 'pred: facing (move)']:.2f} vs parent {S2.loc['parent', 'pred: facing (move)']:.2f})")
if "dp only" in S2.index:
    lost = {k: round(S2.loc["parent", f"own: {k}"] - S2.loc["dp only", f"own: {k}"], 2) for k in ("mob", "terrain")}
    say("H2c the price: poorly predicted content lost in `dp only` (own drop > 0.1)", all(v > 0.1 for v in lost.values()), f"drops {lost}")
if "fade" in S2.index:
    fd = S2.loc["parent", "fresh: facing"] - S2.loc["fade", "fresh: facing"]
    say("H4 control: plain fade erodes (fresh facing drop >= 0.2)", fd >= 0.2, f"drop {fd:.2f}")

## H5: does release follow the predictor?
Per part: how well the parent **predicts** it (decoded predicted frame) against how much a fork **releases** it (parent holds - fork holds, own decoder).
The goal says: what was predicted is kept, what was not is released. Expected: a negative correlation for the dp forks, and none (or erosion everywhere) for
plain `fade`. Twinkle is included as the zero-predictability anchor.

In [ ]:
PARTS = ("agent", "hand", "inventory", "mob", "terrain")
pred_p = pd.Series({k: S2.loc["parent", f"pred: {k}"] for k in PARTS} | {"twinkle": DYN_P.loc["predicted", "twinkle"]})
rel = {}
for n in ("fade", "fade + dp", "fade-r0 + dp", "dp only"):
    if n in S2.index:
        rel[n] = pd.Series({k: S2.loc["parent", f"own: {k}"] - S2.loc[n, f"own: {k}"] for k in PARTS}
                           | {"twinkle": S2.loc["parent", "twinkle: own decoder"] - S2.loc[n, "twinkle: own decoder"]})
REL = pd.DataFrame({"parent predicts": pred_p, **{f"release: {n}": v for n, v in rel.items()}})
display(REL.round(2))
fig, ax = plt.subplots(figsize=(6, 4.5))
for n, v in rel.items():
    rho = pred_p.rank().corr(v.rank())  # Spearman without scipy
    ax.scatter(pred_p, v, color=COLOR[n], label=f"{n} (rho {rho:.2f})")
    if n != "fade":
        say(f"H5 release follows the predictor in `{n}` (rho < -0.5)", rho < -0.5, f"rho {rho:.2f}")
for k in pred_p.index:
    ax.annotate(k, (pred_p[k], max(r[k] for r in rel.values())), fontsize=8)
ax.axhline(0, color="grey", lw=0.5)
ax.set(xlabel="parent: predicted (own decoder)", ylabel="release (parent holds - fork holds)")
ax.legend(fontsize=8)
plt.show()

## Extra fork: an active release (target gradient)
**Question.** All forks above keep the stop-gradient on the target, so nothing *removes* what the predictor cannot produce: the fade only takes away the reward
for it. Earlier evidence says release happens anyway, but slowly (twinkle 0.31 to 0.01 by a fresh decoder, 0.88 to 0.78 by the own decoder). Here the prediction error
also reaches the encoder through the target (`tgt` = 1, ramped in over the same 4000 steps), the LeWM way, **after** availability and with the decoded prediction holding
the content.

| fork | reconstruction | decoded prediction | target gradient | role |
|---|---|---|---|---|
| `fade-r0 + dp` | 1 to 0 | 0 to 1 | off | the reference (above) |
| `fade-r0 + dp + tgt` | 1 to 0 | 0 to 1 | 0 to 1 | the new fork |

**Pre-registered (compared with the parent, and with `fade-r0 + dp`):**
* **E1, faster release:** the noise is released by the usual two of three instruments, and by more than in `fade-r0 + dp` (own-decoder twinkle lower by >= 0.2).
* **E2, content held:** no drop > 0.05 in agent, inventory, facing (fresh decoder) and agent, facing (own decoder), and predicted move-step facing no more than 0.05 below the parent.
* **E3, no collapse:** the latent's effective rank (participation ratio) stays >= 0.7 of the parent's. The open target path collapsed it in notebook 06; this is the risk.

If E1 holds and E2 or E3 fails, the active release costs more than it gives. If E1 fails, the fade plus decoded prediction is as good a release as we have.

In [ ]:
TGT = "fade-r0 + dp + tgt"
FORKS[TGT] = ("fade", "dpr0tgt", FADE, 0.0, 1.0)
FDIR[TGT] = P_J / "forks" / F.fork_name("fade", "dpr0tgt")
print(F.launch_forks(P_J, modes=("fade",), steps=FORK_STEPS, fade=FADE, r_floor=0.0, tag="dpr0tgt", dp=1.0, tgt=1.0))
display(F.status(P_J, modes=[FDIR[TGT].name]))

In [ ]:
import torch


def eff_rank(d):  # participation ratio of the live latent's covariance spectrum
    ck = R.load(d)
    E = R.systems(ck)["live"][0]
    with torch.no_grad():
        z = E(ck.env.sample(2048).x[:, -1])
    ev = torch.linalg.eigvalsh(torch.cov(z.T))
    return float(ev.sum() ** 2 / ev.pow(2).sum())


assert (FDIR[TGT] / "done").exists(), "the extra fork is not done yet"
S2.loc[TGT] = readouts(FDIR[TGT])
rank = {n: eff_rank(P_J if n == "parent" else FDIR[n]) for n in ("parent", "fade-r0 + dp", TGT) if n == "parent" or (FDIR[n] / "done").exists()}
cmp_ = S2.loc[[n for n in ("parent", "fade", "fade-r0 + dp", TGT) if n in S2.index]]
display(cmp_.assign(**{"latent rank": pd.Series(rank)}).round(2))
h, drop = holds(TGT)
r, votes = releases(TGT)
print("drops of learned content vs parent:", drop)
say("E2 holds", h)
say("E2 dynamics kept", dyn_kept(TGT))
say("E1 releases noise (>= 2 of 3)", r, f"votes {votes}")
if "fade-r0 + dp" in S2.index:
    extra = S2.loc["fade-r0 + dp", "twinkle: own decoder"] - S2.loc[TGT, "twinkle: own decoder"]
    say("E1 faster than `fade-r0 + dp` (own-decoder twinkle lower by >= 0.2)", extra >= 0.2, f"difference {extra:.2f}")
say("E3 no collapse (rank >= 0.7 of the parent's)", rank[TGT] >= 0.7 * rank["parent"], f"rank {rank[TGT]:.0f} vs parent {rank['parent']:.0f}")

## Reading the verdicts together
* **H4 first.** If plain `fade` does not erode, the problem did not reproduce and the dp forks are not tested against it.
* **Idea 3** wins if `fade + dp` (or the strict `fade-r0 + dp`) holds, releases the noise, keeps the dynamics, and H5 shows release in order of the predictor.
  If it only matches `continue + dp`, the fade buys nothing and the simpler model (no fade) is the one to keep.
* **Idea 2** (`dp only`) against `fade-r0 + dp`: the same end point, switched at once instead of cross-faded. If they match, the 4k window does not matter.
* **H1** says whether dp is worth keeping at all, independent of the fade.

## Download
Zip `/workspace/results/joint_dp` (checkpoints and `forks/` included) and download it. It is all you need to redo any readout, or to read an earlier step.